# **Ming-3.0-tts for Text To Speech**

Supports multilingual speech, voice cloning, and Paralinguistic Tags.

Github repo: https://github.com/inclusionAI/Ming-3.0-tts

Notebook Source: https://github.com/Isi-dev/Google-Colab_Notebooks

# **Use the 0.5b Model**

In [ ]:
# @title ⚙️ Environment Setup

# 1. Clone the repository
!git clone https://github.com/Isi-dev/Ming-3.0-tts.git
%cd Ming-3.0-tts

# 2. Patch requirements to avoid breaking Colab's native PyTorch bindings
!sed -i '/torch==/d' requirements.txt
!sed -i '/torchaudio==/d' requirements.txt

# 3. Install Linux C++ headers for pynini text normalization
!apt-get update -qq
!apt-get install -y -qq build-essential libopenfst-dev openfst-tools

# 4. Install dependencies
!pip install -r requirements.txt
!pip install pynini==2.1.6
!pip install "huggingface_hub[cli]"

# 5. Download the 0.5B dense model
print("Downloading Ming-3.0-tts-0.5b weights...")
!hf download inclusionAI/Ming-3.0-tts-0.5b --local-dir inclusionAI/Ming-3.0-tts-0.5b


# Initialize Ming-3.0-tts Engine
import sys
import os
import torch
import gc
from IPython.display import Audio, display
from google.colab import files

# Ensure the cloned repo is in the Python path
if '/content/Ming-3.0-tts' not in sys.path:
    sys.path.append('/content/Ming-3.0-tts')

from ming_tts import MingTTS

MODELS = {}

def get_model(repo_id="inclusionAI/Ming-3.0-tts-0.5b"):
    """Loads the model into VRAM safely."""
    if repo_id not in MODELS:
        torch.cuda.empty_cache()
        gc.collect()
        print(f"Loading {repo_id} into VRAM...")

        # Explicitly set text_normalization=False to bypass the missing pynini dependency
        MODELS[repo_id] = MingTTS(repo_id, text_normalization=False)
        print("✅ Successfully loaded!")

    return MODELS[repo_id]

from IPython.display import clear_output
clear_output()
print("✅ Installation and Download complete!")

In [ ]:
# @title 1️⃣ Use Built-In Voices


text_to_synthesize = "The garden will remain open until sunset, provided the weather stays clear." # @param {type:"string"}
# @markdown Select a built-in character voice profile:

selected_profile = "UK Female — Measured Narration (UK_Charlotte)" # @param ["Teenage Girl — AI Assistant (灵小安)", "Teenage Girl — Playful Younger Sister (灵语汐)", "Young Adult Female — Audiobook Narrator (灵小萌)", "Young Adult Female — Customer-Service (灵小柔)", "Young Adult Female — Newsreader (灵小雅)", "Teenage Girl — Child Character (花小呗)", "Young Adult Female — Alluring Character (妩媚妲己)", "Young Adult Male — Caring Boyfriend (灵小锋)", "Young Adult Male — Radio Host (灵小川)", "Young Adult Male — Food Blogger (灵小琪)", "Young Adult Male — Dominant Executive (灵晓声)", "Young Adult Female — Caring Girlfriend (灵风信)", "Young Adult Male — Confident Protagonist (皮六)", "Young Adult Male — Crosstalk Performer (悟空)", "US Female — Assistants & Narration (US_Emma)", "UK Male — Audiobooks (UK_George)", "UK Female — Measured Narration (UK_Charlotte)"]

# Map user-friendly dropdown selection to the exact string required by the engine
ip_mapping = {
    # --- Chinese Voices (Without Emotion Variants) ---[cite: 1]
    "Teenage Girl — AI Assistant (灵小安)": "灵小安",
    "Teenage Girl — Playful Younger Sister (灵语汐)": "灵语汐",
    "Young Adult Female — Audiobook Narrator (灵小萌)": "灵小萌",
    "Young Adult Female — Customer-Service (灵小柔)": "灵小柔",
    "Young Adult Female — Newsreader (灵小雅)": "灵小雅",
    "Teenage Girl — Child Character (花小呗)": "花小呗",
    "Young Adult Female — Alluring Character (妩媚妲己)": "妩媚妲己",
    "Young Adult Male — Caring Boyfriend (灵小锋)": "灵小锋",
    "Young Adult Male — Radio Host (灵小川)": "灵小川",
    "Young Adult Male — Food Blogger (灵小琪)": "灵小琪",
    "Young Adult Male — Confident Protagonist (皮六)": "皮六",
    "Young Adult Male — Crosstalk Performer (悟空)": "悟空",

    # --- Chinese Voices (Base Voices for Emotion Variants) ---[cite: 1]
    "Young Adult Male — Dominant Executive (灵晓声)": "灵晓声",
    "Young Adult Female — Caring Girlfriend (灵风信)": "灵风信",

    # --- English Voices ---[cite: 1]
    "US Female — Assistants & Narration (US_Emma)": "US_Emma",
    "UK Male — Audiobooks (UK_George)": "UK_George",
    "UK Female — Measured Narration (UK_Charlotte)": "UK_Charlotte"
}

# Fetch the exact IP string based on the user's selection
ip_voice = ip_mapping.get(selected_profile, "灵小安")

out_path = "output_ip.wav"

model = get_model()
model.synthesize(
    task="IP",
    text="The garden will remain open until sunset, provided the weather stays clear.",
    controls={"ip": ip_voice},
    use_zero_spk_emb=True,
    output_path=out_path
)

display(Audio(out_path))

In [ ]:
# @title 2️⃣ Zero-Shot Clone & Paralinguistic Tags
# @markdown Add tags like `[Laughter]`, `[Sigh]`, `[Breathing]`, `[Cough]`, `[Surprise-ah]`, `[Surprise-oh]`, `[Dissatisfaction-hnn]`, `[Uhm]`, `[Confirmation-en]`, `[Question-ei]`, `[Question-ah]`, `[Question-en]`, `[Question-oh]`, `[Question-huh]` directly into the text.
text_to_synthesize = "Well [Sigh], I guess we can try that [Laughter]." # @param {type:"string"}
language = "English" # @param ["Chinese", "English", "Japanese", "Korean", "German", "French", "Spanish"]
reference_text = "Enter the transcript of your reference audio." # @param {type:"string"}

print("Please upload a 3-10 second reference WAV file...")
uploaded = files.upload()

if uploaded:
    ref_audio_path = list(uploaded.keys())[0]
    out_path = "output_tts.wav"

    model = get_model()
    model.synthesize(
        task="TTS",
        language_id=language,
        text=text_to_synthesize,
        prompt_wav_path=ref_audio_path,
        prompt_text=reference_text,
        use_spk_emb=True,
        output_path=out_path
    )

    display(Audio(out_path))
    os.remove(ref_audio_path)

In [ ]:
# @title 3️⃣ Emotion Control
text_to_synthesize = "I can't believe we actually pulled this off!" # @param {type:"string"}
language = "English" # @param ["Chinese", "English"]
target_emotion = "中性 (Neutral)" # @param ["高兴 (Happy)", "悲伤 (Sad)", "愤怒 (Angry)", "恐惧 (Fear)", "厌恶 (Disgust)", "惊讶 (Surprise)", "中性 (Neutral)"]
reference_text = "Enter the transcript of your reference audio." # @param {type:"string"}

print("Please upload your neutral reference WAV file...")
uploaded = files.upload()

if uploaded:
    ref_audio_path = list(uploaded.keys())[0]
    emotion_kw = target_emotion.split(" ")[0] # Extracts the Chinese dictionary keyword
    out_path = "output_emotion.wav"

    model = get_model()
    # Structured tasks require the 'controls' dictionary
    model.synthesize(
        task="EMOTION",
        controls={"emotion": emotion_kw},
        language_id=language,
        text=text_to_synthesize,
        prompt_wav_path=ref_audio_path,
        prompt_text=reference_text,
        use_spk_emb=True,
        output_path=out_path
    )

    display(Audio(out_path))
    os.remove(ref_audio_path)

In [ ]:
# @title 4️⃣ Voice Design
# @markdown Creates a character voice purely from a text description without needing reference audio.
text_to_synthesize = "Once upon a time, in a kingdom far away, lived a tiny dragon." # @param {type:"string"}
language = "English" # @param ["Chinese", "English"]
voice_style = "A warm, elderly grandfather reading a bedtime story" # @param {type:"string"}

out_path = "output_voice_design.wav"

model = get_model()
# Voice design uses 'controls={"style": ...}' and no reference audio inputs
model.synthesize(
    task="VOICE_DESIGN",
    controls={"style": voice_style},
    language_id=language,
    text=text_to_synthesize,
    use_spk_emb=False,
    output_path=out_path
)

display(Audio(out_path))

In [ ]:
# @title 5️⃣ Freeform Instruction
# @markdown Guides the delivery style using an unstructured natural language instruction.
text_to_synthesize = "I told you to leave the room! Get out!" # @param {type:"string"}
language = "English" # @param ["Chinese", "English"]
instruction = "Speak in a very loud, panicked, and angry voice" # @param {type:"string"}
reference_text = "Enter the transcript of your reference audio." # @param {type:"string"}

print("Please upload your reference WAV file...")
uploaded = files.upload()

if uploaded:
    ref_audio_path = list(uploaded.keys())[0]
    out_path = "output_freeform.wav"

    model = get_model()
    # Freeform uses 'instruction' directly and forbids the 'controls' dict
    model.synthesize(
        task="FREEFORM",
        instruction=instruction,
        language_id=language,
        text=text_to_synthesize,
        prompt_wav_path=ref_audio_path,
        prompt_text=reference_text,
        use_spk_emb=True,
        output_path=out_path
    )

    display(Audio(out_path))
    os.remove(ref_audio_path)

In [ ]:
# @title 6️⃣ Basic Controls
text_to_synthesize = "This is playing at a fast speed with a high pitch." # @param {type:"string"}
language = "English" # @param ["Chinese", "English"]
speed = "快速 (Fast)" # @param ["慢速 (Slow)", "中速 (Medium)", "快速 (Fast)"]
pitch = "低 (Low)" # @param ["低 (Low)", "中 (Medium)", "高 (High)"]
volume = "低 (Low)" # @param ["低 (Low)", "中 (Medium)", "高 (High)"]
reference_text = "Enter the transcript of your reference audio." # @param {type:"string"}

print("Please upload your reference WAV file...")
uploaded = files.upload()

if uploaded:
    ref_audio_path = list(uploaded.keys())[0]
    out_path = "output_basic.wav"

    model = get_model()
    # Basic task requires a strict dict mapping of speed, pitch, and volume
    model.synthesize(
        task="BASIC",
        controls={
            "speed": speed.split(" ")[0],
            "pitch": pitch.split(" ")[0],
            "volume": volume.split(" ")[0]
        },
        language_id=language,
        text=text_to_synthesize,
        prompt_wav_path=ref_audio_path,
        prompt_text=reference_text,
        use_spk_emb=True,
        output_path=out_path
    )

    display(Audio(out_path))
    os.remove(ref_audio_path)

# **Use the Tiny Model**
- A pop up will appear during the installation instructing you to restart the session. Please let the cell finish running before you restart.

In [ ]:
# @title ⚙️ Installation for Ming-3.0-tts-tiny

import os
import time

print("1️⃣ Cloning repository...")
if not os.path.exists('/content/Ming-3.0-tts'):
    !git clone https://github.com/Isi-dev/Ming-3.0-tts.git
%cd /content/Ming-3.0-tts

# print("\n2️⃣ Removing TensorFlow, JAX, and vision packages...")
# # Purging these prevents transformers from triggering TF/JAX imports
# !pip uninstall -y tensorflow jax jaxlib torchvision xformers fastai

print("\n2️⃣ Surgically editing requirements-tiny.txt...")
# Delete the rigid versions that break Colab's Python 3.13 environment
!sed -i '/^torch==/d' requirements-tiny.txt
!sed -i '/^torchaudio==/d' requirements-tiny.txt
!sed -i '/^numpy==/d' requirements-tiny.txt
!sed -i '/^scipy==/d' requirements-tiny.txt
!sed -i '/^triton==/d' requirements-tiny.txt

print("\n3️⃣ Installing patched requirements...")
!pip install -r requirements-tiny.txt
!pip install "huggingface_hub[cli]"

print("\n4️⃣ Compiling fla-core natively...")
!pip install -U fla-core

print("\n Downloading Ming-3.0-tts-tiny weights...")
!hf download inclusionAI/Ming-3.0-tts-tiny --local-dir inclusionAI/Ming-3.0-tts-tiny

from IPython.display import clear_output
clear_output()

# print("\n🔄 Setup Complete! Restarting runtime to initialize CUDA/C++ bindings...")
# print("⚠️ It is normal to see a 'Session crashed' popup. Just close it and run the next code cell.")
# time.sleep(2)
# os.kill(os.getpid(), 9)

clear_output()
print("✅ Installation and Download complete!")



In [ ]:

# @title 🧬 Initialize MingTinyTTS Engine

import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TORCH"] = "1"

import sys
import torch
import gc

if '/content/Ming-3.0-tts' not in sys.path:
    sys.path.append('/content/Ming-3.0-tts')

from ming_tiny import MingTinyTTS

TINY_MODELS = {}

def get_tiny_model(model_path="inclusionAI/Ming-3.0-tts-tiny"):
    if model_path not in TINY_MODELS:
        torch.cuda.empty_cache()
        gc.collect()
        print(f"Loading {model_path} into VRAM...")

        # Explicitly disable text normalization to bypass pynini
        TINY_MODELS[model_path] = MingTinyTTS.from_pretrained(
            model_path,
            device="cuda:0",
            text_normalization=False
        )
        print("✅ MingTinyTTS successfully loaded!")

    return TINY_MODELS[model_path]

# Load into VRAM
tts = get_tiny_model()
from IPython.display import clear_output
clear_output()
print("✅ Preparation complete!")

In [ ]:
# @title 1️⃣ Voice Design + Podcast Tasks
from IPython.display import Audio, display
import os

# @markdown ### Character Design
# @markdown Describe the voice styles for your two speakers.
spk1_style = "A deep, professional male radio host with a smooth, calm delivery" # @param {type:"string"}
spk2_style = "A bright, energetic female tech journalist" # @param {type:"string"}

# @markdown ---
# @markdown ### 📝 Script Editor
line_1_speaker1 = "Welcome back to the show. Today, we're talking about open-source AI." # @param {type:"string"}
line_2_speaker2 = "Thanks for having me! It's incredible how fast the technology is moving right now." # @param {type:"string"}
line_3_speaker1 = "It really is. Can we generate this entire conversation without uploading a single audio file?" # @param {type:"string"}
line_4_speaker2 = "Absolutely. We just design the voices on the fly and stitch them together." # @param {type:"string"}

# tts = get_tiny_model()

# STEP 1: Generate the reference audio for Speaker 1
print(f"Designing Speaker 1: {spk1_style}...")
spk1_ref_text = "This is the reference voice for speaker one."
spk1_ref_path = "temp_spk1_ref.wav"
tts.generate(
    task="voice_design",
    text=spk1_ref_text,
    style=spk1_style,
    output_path=spk1_ref_path
)

# STEP 2: Generate the reference audio for Speaker 2
print(f"Designing Speaker 2: {spk2_style}...")
spk2_ref_text = "And this is the reference voice for speaker two."
spk2_ref_path = "temp_spk2_ref.wav"
tts.generate(
    task="voice_design",
    text=spk2_ref_text,
    style=spk2_style,
    output_path=spk2_ref_path
)

# STEP 3: Compile the script
script_lines = []
if line_1_speaker1.strip(): script_lines.append(f"speaker_1: {line_1_speaker1}")
if line_2_speaker2.strip(): script_lines.append(f"speaker_2: {line_2_speaker2}")
if line_3_speaker1.strip(): script_lines.append(f"speaker_1: {line_3_speaker1}")
if line_4_speaker2.strip(): script_lines.append(f"speaker_2: {line_4_speaker2}")
script_text = "\n".join(script_lines)

print("\n⚙️ Synthesizing the final podcast conversation...")

# STEP 4: Feed the generated voices into the podcast task
out_path = "output_designed_podcast.wav"
tts.generate(
    task="podcast",
    text=script_text,
    prompt_wav=[spk1_ref_path, spk2_ref_path],
    prompt_text=[spk1_ref_text, spk2_ref_text],
    output_path=out_path
)

print("\n✅ Podcast generated successfully!")
display(Audio(out_path))

# Cleanup temporary reference files
os.remove(spk1_ref_path)
os.remove(spk2_ref_path)

In [ ]:
# @title 2️⃣ Multi-Speaker Podcast (voices from reference)
from google.colab import files
from IPython.display import Audio, display
import os

# @markdown ### 📝 Script Editor
# @markdown Fill in the dialogue below. Leave a line blank to skip it.
line_1_speaker1 = "Did you hear about the new open-source voice models?" # @param {type:"string"}
line_2_speaker2 = "Yes! The multi-speaker generation sounds surprisingly natural." # @param {type:"string"}
line_3_speaker1 = "Exactly, it handles the conversational back-and-forth really well." # @param {type:"string"}
line_4_speaker2 = "I wonder if we can run it locally on a standard GPU." # @param {type:"string"}

# @markdown ---
# @markdown ### Speaker Configuration
spk1_transcript = "Enter the transcript of your reference audio for speaket 1." # @param {type:"string"}
spk2_transcript = "Enter the transcript of your reference audio for speaker 2." # @param {type:"string"}

# 1. Combine the variables into the strict format required by Ming-3.0-tts-tiny
script_lines = []
if line_1_speaker1.strip(): script_lines.append(f"speaker_1: {line_1_speaker1}")
if line_2_speaker2.strip(): script_lines.append(f"speaker_2: {line_2_speaker2}")
if line_3_speaker1.strip(): script_lines.append(f"speaker_1: {line_3_speaker1}")
if line_4_speaker2.strip(): script_lines.append(f"speaker_2: {line_4_speaker2}")

script_text = "\n".join(script_lines)

print("Script compiled as:\n")
print(script_text)
print("\n-----------------------------------\n")

print("1️⃣ Upload Speaker 1 WAV...")
uploaded_spk1 = files.upload()

print("\n2️⃣ Upload Speaker 2 WAV...")
uploaded_spk2 = files.upload()

if uploaded_spk1 and uploaded_spk2:
    spk1_path = list(uploaded_spk1.keys())[0]
    spk2_path = list(uploaded_spk2.keys())[0]

    out_path = "output_podcast.wav"
    # tts = get_tiny_model()

    # 2. Pass the compiled script and ordered arrays to the generator
    tts.generate(
        task="podcast",
        text=script_text,
        prompt_wav=[spk1_path, spk2_path],
        prompt_text=[spk1_transcript, spk2_transcript],
        output_path=out_path
    )

    print("\n✅ Podcast generated successfully:")
    display(Audio(out_path))

    # Cleanup
    os.remove(spk1_path)
    os.remove(spk2_path)
else:
    print("❌ Failed: Both speaker audio references are required.")

In [ ]:
# @title 3️⃣ Zero-Shot Cloning (seed)
text_to_synthesize = "The garden will remain open until sunset, provided the weather stays clear." # @param {type:"string"}
language = "English" # @param ["Chinese", "English"]
reference_text = "Enter the transcript of your reference audio." # @param {type:"string"}

from google.colab import files
from IPython.display import Audio, display
import os

print("Please upload a 3-10 second reference WAV file...")
uploaded = files.upload()

if uploaded:
    ref_audio_path = list(uploaded.keys())[0]
    out_path = "output_seed.wav"

    # tts = get_tiny_model()
    tts.generate(
        task="seed",
        text=text_to_synthesize,
        target_language=language,
        prompt_wav=ref_audio_path,
        prompt_text=reference_text,
        output_path=out_path
    )

    display(Audio(out_path))
    os.remove(ref_audio_path)

In [ ]:
# @title 4️⃣ Paralinguistic Tags (paralinguistic)
text_to_synthesize = "I checked the drawer twice [Sigh] and it was empty [Uhm] again. What shall I do now?" # @param {type:"string"}
language = "English" # @param ["Chinese", "English"]
reference_text = "Enter the transcript of your reference audio." # @param {type:"string"}

print("Please upload your reference WAV file...")
uploaded = files.upload()

if uploaded:
    ref_audio_path = list(uploaded.keys())[0]
    out_path = "output_paralinguistic.wav"

    # tts = get_tiny_model()
    tts.generate(
        task="paralinguistic",
        text=text_to_synthesize,
        target_language=language,
        prompt_wav=ref_audio_path,
        prompt_text=reference_text,
        output_path=out_path
    )

    display(Audio(out_path))
    os.remove(ref_audio_path)

In [ ]:
# @title 5️⃣ Cross-Lingual Synthesis (multilingual)
text_to_synthesize = "Bonjour, je suis heureux de vous rencontrer aujourd'hui." # @param {type:"string"}
language = "French" # @param ["German", "Spanish", "French", "Korean", "Japanese"]
reference_text = "Enter the transcript of your reference audio." # @param {type:"string"}

print("Please upload your reference WAV file...")
uploaded = files.upload()

if uploaded:
    ref_audio_path = list(uploaded.keys())[0]
    out_path = "output_multilingual.wav"

    # tts = get_tiny_model()
    tts.generate(
        task="multilingual",
        text=text_to_synthesize,
        target_language=language,
        prompt_wav=ref_audio_path,
        prompt_text=reference_text,
        output_path=out_path
    )

    display(Audio(out_path))
    os.remove(ref_audio_path)

In [ ]:
# @title 6️⃣ Emotion Control (emotion)
text_to_synthesize = "You completed the entire restoration in a single night? I can hardly believe it!" # @param {type:"string"}
target_emotion = "高兴 (Happy)" # @param ["高兴 (Happy)", "悲伤 (Sad)", "愤怒 (Angry)", "恐惧 (Fear)", "厌恶 (Disgust)", "惊讶 (Surprise)", "中性 (Neutral)"]

print("Please upload a neutral reference WAV file...")
uploaded = files.upload()

if uploaded:
    ref_audio_path = list(uploaded.keys())[0]
    emotion_kw = target_emotion.split(" ")[0]
    out_path = "output_emotion.wav"

    # tts = get_tiny_model()
    tts.generate(
        task="emotion",
        text=text_to_synthesize,
        emotion=emotion_kw,
        prompt_wav=ref_audio_path,
        prompt_text=None,
        output_path=out_path
    )

    display(Audio(out_path))
    os.remove(ref_audio_path)

In [ ]:
# @title 7️⃣ Basic Attributes (basic)
text_to_synthesize = "This sentence will be read very quickly with a high pitch." # @param {type:"string"}
speed = "慢速 (Slow)" # @param ["慢速 (Slow)", "中速 (Medium)", "快速 (Fast)"]
pitch = "\u9ad8 (High)" # @param ["低 (Low)", "中 (Medium)", "高 (High)"]
volume = "\u9ad8 (High)" # @param ["低 (Low)", "中 (Medium)", "高 (High)"]

print("Please upload a reference WAV file...")
uploaded = files.upload()

if uploaded:
    ref_audio_path = list(uploaded.keys())[0]
    out_path = "output_basic.wav"

    # tts = get_tiny_model()
    tts.generate(
        task="basic",
        text=text_to_synthesize,
        speed=speed.split(" ")[0],
        pitch=pitch.split(" ")[0],
        volume=volume.split(" ")[0],
        prompt_wav=ref_audio_path,
        prompt_text=None,
        output_path=out_path
    )

    display(Audio(out_path))
    os.remove(ref_audio_path)

In [ ]:
# @title 8️⃣ Director Instructions (freeform) {"single-column":true}
text_to_synthesize = "You changed the plan without telling anyone. How were we supposed to know?" # @param {type:"string"}
instruction_text = "Sound angry but controlled. Stress the accusation without shouting." # @param {type:"string"}

print("Please upload a reference WAV file...")
uploaded = files.upload()

if uploaded:
    ref_audio_path = list(uploaded.keys())[0]
    out_path = "output_freeform.wav"

    # tts = get_tiny_model()
    tts.generate(
        task="freeform",
        text=text_to_synthesize,
        instruction=instruction_text,
        prompt_wav=ref_audio_path,
        prompt_text=None,
        output_path=out_path
    )

    display(Audio(out_path))
    os.remove(ref_audio_path)

In [ ]:
# @title 9️⃣ Voice Design (voice_design) {"single-column":true}
text_to_synthesize = "Beyond the northern ridge lies a city that vanished from every map." # @param {type:"string"}
style_description = "A mysterious fantasy narrator with a low pitch and dramatic pauses." # @param {type:"string"}

out_path = "output_voice_design.wav"

# tts = get_tiny_model()
# No prompt_wav or prompt_text is passed for Voice Design
tts.generate(
    task="voice_design",
    text=text_to_synthesize,
    style=style_description,
    output_path=out_path
)

display(Audio(out_path))

In [ ]:
# @title 🔟 Built-In IP Voices (ip) {"single-column":true}
text_to_synthesize = "The garden will remain open until sunset, provided the weather stays clear." # @param {type:"string"}
# @markdown Select a built-in character voice profile:

selected_profile = "Young Adult Female — Caring Girlfriend (灵风信)" # @param ["Teenage Girl — AI Assistant (灵小安)", "Teenage Girl — Playful Younger Sister (灵语汐)", "Young Adult Female — Audiobook Narrator (灵小萌)", "Young Adult Female — Customer-Service (灵小柔)", "Young Adult Female — Newsreader (灵小雅)", "Teenage Girl — Child Character (花小呗)", "Young Adult Female — Alluring Character (妩媚妲己)", "Young Adult Male — Caring Boyfriend (灵小锋)", "Young Adult Male — Radio Host (灵小川)", "Young Adult Male — Food Blogger (灵小琪)", "Young Adult Male — Dominant Executive (灵晓声)", "Young Adult Female — Caring Girlfriend (灵风信)", "Young Adult Male — Confident Protagonist (皮六)", "Young Adult Male — Crosstalk Performer (悟空)", "US Female — Assistants & Narration (US_Emma)", "UK Male — Audiobooks (UK_George)", "UK Female — Measured Narration (UK_Charlotte)"]

# Map user-friendly dropdown selection to the exact string required by the engine
ip_mapping = {
    # --- Chinese Voices (Without Emotion Variants) ---[cite: 1]
    "Teenage Girl — AI Assistant (灵小安)": "灵小安",
    "Teenage Girl — Playful Younger Sister (灵语汐)": "灵语汐",
    "Young Adult Female — Audiobook Narrator (灵小萌)": "灵小萌",
    "Young Adult Female — Customer-Service (灵小柔)": "灵小柔",
    "Young Adult Female — Newsreader (灵小雅)": "灵小雅",
    "Teenage Girl — Child Character (花小呗)": "花小呗",
    "Young Adult Female — Alluring Character (妩媚妲己)": "妩媚妲己",
    "Young Adult Male — Caring Boyfriend (灵小锋)": "灵小锋",
    "Young Adult Male — Radio Host (灵小川)": "灵小川",
    "Young Adult Male — Food Blogger (灵小琪)": "灵小琪",
    "Young Adult Male — Confident Protagonist (皮六)": "皮六",
    "Young Adult Male — Crosstalk Performer (悟空)": "悟空",

    # --- Chinese Voices (Base Voices for Emotion Variants) ---[cite: 1]
    "Young Adult Male — Dominant Executive (灵晓声)": "灵晓声",
    "Young Adult Female — Caring Girlfriend (灵风信)": "灵风信",

    # --- English Voices ---[cite: 1]
    "US Female — Assistants & Narration (US_Emma)": "US_Emma",
    "UK Male — Audiobooks (UK_George)": "UK_George",
    "UK Female — Measured Narration (UK_Charlotte)": "UK_Charlotte"
}

# Fetch the exact IP string based on the user's selection
ip_voice = ip_mapping.get(selected_profile, "灵小安")

out_path = "output_ip.wav"

# tts = get_tiny_model()
# No prompt_wav or prompt_text is passed for IP Voices
tts.generate(
    task="ip",
    text=text_to_synthesize,
    ip=ip_voice,
    output_path=out_path
)

display(Audio(out_path))